# 6.5.1 State Space Models (SSMs)

Implement a linear SSM with A, B, C, D matrices. Apply bilinear discretisation and run on a noisy sine-wave input.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)
T = 200; t = np.linspace(0, 4*np.pi, T)
u = np.sin(t) + 0.3*rng.standard_normal(T)
N = 4
A_c = -np.diag([0.5, 1.0, 2.0, 4.0])
B_c = rng.standard_normal(N)
C = rng.standard_normal(N)
D = 0.1

def discretise(Ac, Bc, dt):
    I = np.eye(N)
    Ad = np.linalg.solve(I - dt/2*Ac, I + dt/2*Ac)
    Bd = np.linalg.solve(I - dt/2*Ac, dt*Bc)
    return Ad, Bd

def run_ssm(u, A, B, C, D):
    h = np.zeros(N); ys = []
    for x in u:
        h = A@h + B*x; ys.append(float(C@h + D*x))
    return np.array(ys)

Ad, Bd = discretise(A_c, B_c, 4*np.pi/T)
y = run_ssm(u, Ad, Bd, C, D)
print(f'State dim: {N}, T: {T}')
print(f'Eigenvalues: {np.linalg.eigvals(Ad).real.round(4)}')

In [ ]:
impulse = np.zeros(T); impulse[0] = 1.0
y_imp = run_ssm(impulse, Ad, Bd, C, D)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(t, u, color='lightgray', lw=1, label='Input')
axes[0].plot(t, y, color='steelblue', lw=2, label='SSM Output')
axes[0].set(xlabel='t', ylabel='Signal', title='SSM: Input vs Output'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(t[:60], y_imp[:60], color='tomato', lw=2)
axes[1].set(xlabel='t', ylabel='y(t)', title='Impulse Response'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/state_space.png')
print('Saved state_space.png')